# Классификация звёздных объектов (SDSS)

**Тема:** Многоклассовая классификация  
**Инструменты:** `pandas`, `RandomForestClassifier`

Классифицируем астрономические объекты на 3 класса по спектральным измерениям обзора Sloan Digital Sky Survey (SDSS).

| Класс | Описание |
|---|---|
| **GALAXY** | Далёкая галактика — миллиарды звёзд |
| **QSO** | Квазар — крайне яркое ядро активной галактики |
| **STAR** | Отдельная звезда в нашей галактике |

**Признаки:**
- `u, g, r, i, z` — фотометрические величины в 5 диапазонах длин волн
- `redshift` (красное смещение) — самый разделяющий признак: у звёзд ≈ 0, у галактик 0.1–1.0, у квазаров 1.0+
- `spectral_type`, `galaxy_population` — категориальные признаки

---
## Шаги
1. Загрузка данных
2. Кодирование категориальных признаков
3. Обучение RandomForest
4. Оценка — balanced accuracy + CV + важность признаков
5. Генерация сабмита

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import balanced_accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix

train = pd.read_csv('/kaggle/input/predicting-stellar-class/train.csv')
test  = pd.read_csv('/kaggle/input/predicting-stellar-class/test.csv')

print('Размер train:', train.shape)
print('Размер test: ', test.shape)
train.head()

In [ ]:
# Распределение классов
counts = train['class'].value_counts()
print(counts)
print()
print((counts / len(train) * 100).round(1).astype(str) + '%')

## 2. Кодирование категориальных признаков

`spectral_type` и `galaxy_population` — текстовые колонки. Переводим их в числа через `LabelEncoder`.

In [ ]:
le_spec = LabelEncoder()
train['spectral_type_encoded'] = le_spec.fit_transform(train['spectral_type'])
test['spectral_type_encoded']  = le_spec.transform(test['spectral_type'])

le_gal = LabelEncoder()
train['galaxy_population_encoded'] = le_gal.fit_transform(train['galaxy_population'])
test['galaxy_population_encoded']  = le_gal.transform(test['galaxy_population'])

features = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z',
            'redshift', 'spectral_type_encoded', 'galaxy_population_encoded']

X_train = train[features]
y_train = train['class']
X_test  = test[features]

print('Признаки:', features)
print('Размер X_train:', X_train.shape)

## 3. Обучение модели

**RandomForestClassifier** с `max_depth=15` — достаточно глубокий, чтобы уловить сложные границы между GALAXY/QSO/STAR.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

print("Обучение модели, подождите...")
model.fit(X_train, y_train)
print("Готово.")

## 4. Оценка качества

**Balanced accuracy** — в отличие от обычной accuracy, усредняет полноту (recall) по классам. Это важно, потому что классы несбалансированы (GALAXY 65%, QSO 20%, STAR 14%).

In [ ]:
print("Запуск кросс-валидации...")
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='balanced_accuracy', n_jobs=-1)

y_pred_train = model.predict(X_train)
train_acc = balanced_accuracy_score(y_train, y_pred_train)

print('=' * 50)
print(' МЕТРИКИ (Balanced Accuracy):')
print(f'  • Train:                 {train_acc:.4f}')
print(f'  • Кросс-валидация (CV):  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 50)

In [ ]:
# Метрики по каждому классу
print(classification_report(y_train, y_pred_train))

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_train, y_pred_train, labels=['GALAXY', 'QSO', 'STAR'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['GALAXY', 'QSO', 'STAR'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Матрица ошибок (train)')
plt.tight_layout()
plt.show()

In [ ]:
importances = sorted(zip(features, model.feature_importances_), key=lambda x: x[1], reverse=True)

print('=' * 50)
print(' ВАЖНОСТЬ ПРИЗНАКОВ (по убыванию):')
for name, imp in importances:
    print(f'  • {name:<28}: {imp:.2%}')
print('=' * 50)

names, vals = zip(*importances)
plt.figure(figsize=(10, 4))
plt.bar(names, vals)
plt.title('Важность признаков')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Генерация сабмита

In [ ]:
print("Генерация предсказаний для test...")
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'id':    test['id'],
    'class': predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv успешно сохранён!')
print(submission['class'].value_counts())
submission.head()